# Pipeline NILM — experimentación en Colab

Notebook delgado: toda la lógica vive en el paquete `nilm_pipeline/` (repo `NILM_pipeline`).
Cada experimento es una instancia de `Config`; los resultados se acumulan en `outputs/leaderboard.csv`.

**Fuentes (GitHub, repos privados)**:
- código: `github.com/klopszteinleonel/NILM_pipeline`
- datos:  `github.com/klopszteinleonel/NILM_local_dataset` (una carpeta por clase, `combinadas/`, `nada/`)

**Antes de correr en Colab**: crear un *secret* llamado `GITHUB_TOKEN` (ícono de la llave, barra lateral)
con un token de GitHub con permiso de lectura sobre los dos repos, y habilitarlo para este notebook.
La celda 0 clona ambos repos en `/content/` (o hace `git pull` si ya están) e instala dependencias.
`outputs/` en Colab es efímero: para conservar el leaderboard entre sesiones poné `PERSISTIR_EN_DRIVE = True`.

Principio rector: todo lo que hace el pipeline tiene que poder correr en el ESP32-S3 (ver `AGENTS.md`).

In [ ]:
# 0. Setup: en Colab clona los repos de GitHub; en local usa rutas relativas
import os, sys, subprocess

REPO_PIPELINE = "klopszteinleonel/NILM_pipeline"
REPO_DATOS    = "klopszteinleonel/NILM_local_dataset"
PERSISTIR_EN_DRIVE = False     # True: outputs/ en MyDrive/NILM/outputs (sobrevive a la sesión)

def clonar_o_actualizar(repo, destino, token):
    url = f"https://{'x-access-token:' + token + '@' if token else ''}github.com/{repo}.git"
    if os.path.isdir(os.path.join(destino, ".git")):
        cmd = ["git", "-C", destino, "pull", "--ff-only", "-q"]
    else:
        cmd = ["git", "clone", "-q", "--depth", "1", url, destino]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        detalle = r.stderr.replace(token, "<token>") if token else r.stderr
        raise RuntimeError(f"git falló para {repo}: {detalle.strip()} -- "
                           "Si dice 'not found': el token no tiene acceso a ese repo "
                           "(Repository access -> incluir los dos repos, Contents: Read-only).")
    print(f"{repo} -> {destino}")

try:
    from google.colab import userdata
    try:
        TOKEN = userdata.get("GITHUB_TOKEN")
    except Exception:
        TOKEN = ""          # sin secret: solo funciona si los repos son públicos
        print("[AVISO] No hay secret GITHUB_TOKEN; se intenta clonar sin token")
    RUTA_PIPELINE = "/content/NILM_pipeline"
    RUTA_DATOS    = "/content/NILM_local_dataset"
    clonar_o_actualizar(REPO_PIPELINE, RUTA_PIPELINE, TOKEN)
    clonar_o_actualizar(REPO_DATOS, RUTA_DATOS, TOKEN)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    f"{RUTA_PIPELINE}/requirements-colab.txt"], check=True)
    if PERSISTIR_EN_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        RUTA_SALIDAS = "/content/drive/MyDrive/NILM/outputs"
    else:
        RUTA_SALIDAS = os.path.join(RUTA_PIPELINE, "outputs")
    EN_COLAB = True
except ImportError:
    RUTA_PIPELINE = os.path.abspath(os.path.join(os.getcwd(), ".."))
    RUTA_DATOS    = os.path.abspath(os.path.join(RUTA_PIPELINE, "..", "clienteTCP", "capturas_limpio"))
    RUTA_SALIDAS  = os.path.join(RUTA_PIPELINE, "outputs")
    EN_COLAB = False
os.makedirs(RUTA_SALIDAS, exist_ok=True)
sys.path.insert(0, RUTA_PIPELINE)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import nilm_pipeline
from nilm_pipeline import *
from nilm_pipeline import plots
print(f"nilm_pipeline {nilm_pipeline.__version__} | colab={EN_COLAB}\ndatos:   {RUTA_DATOS}\nsalidas: {RUTA_SALIDAS}")

## 1. Ingesta
Escanea la carpeta de capturas, arma el manifest (con hash del dataset) y una cache `.npz`.
Para agregar datos: copiar los JSON a la carpeta que corresponda y volver a correr esta celda.

In [ ]:
ds = cargar_dataset(RUTA_DATOS)
print(ds.resumen())
guardar_manifest(ds.manifest_filas, os.path.join(RUTA_SALIDAS, "manifest.csv"))
guardar_cache(ds, os.path.join(RUTA_SALIDAS, "cache_dataset.npz"))
fs = ds.fs()
sen, Y, G = ds.senales(), ds.y(), ds.grupos()

## 2. Configuración base
`Config` es la única fuente de verdad. Base: **FFT cruda** (solo los 614 bins directos de la FFT de W=2048 hasta 3 kHz),
sin augment, 100 mezclas sintéticas por fold con fundamentales exactamente en fase (solo para entrenar), OvR de RandomForest 20×8.
El test son siempre capturas reales: individuales (folds) y combinadas (fuera de los folds).
`AGRUPAR` fusiona clases físicamente indistinguibles (resistivas puras de ~6 A: misma amplitud, THD ≈ 0.02);
dejalo vacío (`{}`) para evaluarlas por separado.

In [ ]:
AGRUPAR = {"calefon": "resistiva", "horno": "resistiva", "pava": "resistiva", "plancha": "resistiva"}
CFG = Config(agrupar=AGRUPAR)
print(CFG.nombre())
pd.Series({k: v for k, v in CFG.to_dict().items() if k != "bandas_hz"}).to_frame("valor").T

## 3. Sanity checks visuales
Señal, detalle y espectro (con las armónicas de la f0 estimada) de una captura por clase; una mezcla sintética; augment.

In [ ]:
for clase in ("aire", "tele", "calefon"):
    r = next((r for r in ds.registros if r.clases == (clase,)), None)
    if r is not None:
        plots.plot_captura(r.datos, fs, CFG, titulo=f"{clase} — {r.ruta}")
plt.show()

In [ ]:
rng = np.random.default_rng(CFG.seed)
m = sintetizar_combinadas(sen, Y, G, ds.clases, CFG, rng, total=3, fs=fs)
i = 0
origen = [j for j in range(len(G)) if G[j] in m.origenes[i]]
print("mezcla 0 =", [c for c, b in zip(ds.clases, m.Y[i]) if b], "| fuentes:", m.origenes[i])
plots.plot_mix([sen[j] for j in origen], [G[j].split('/')[-1][:24] for j in origen], m.senales[i], fs)
V = segmentar(sen[0], CFG.W, CFG.solape)
Vp, esc = preprocesar_ventanas(V, CFG)
plots.plot_augment(Vp[0], CFG, fs)
plt.show()

## 4. Features
Vector `rawfft` de una ventana por clase (espectro directo) y, como referencia, la mediana por clase de las features `compact`.
La amplitud absoluta (`rms_A`, `h1_A`) es lo único que separa a las resistivas puras entre sí.

In [ ]:
X, Yw, Gw, idx = segmentar_dataset(sen, Y, G, CFG)
Vp, esc = preprocesar_ventanas(X, CFG)
F, nombres = extraer_features(Vp, esc, CFG, fs)
print(f"rawfft: {len(F)} ventanas x {len(nombres)} features ({nombres[0]} … {nombres[-1]})")
fig, ax = plt.subplots(figsize=(12, 3.5))
freqs = np.array([float(n[4:-2]) for n in nombres])
for clase in ("aire", "tele", "lavarropa", "calefon"):
    i = next((k for k in range(len(Yw)) if Yw[k].sum() == 1 and ds.clases[Yw[k].argmax()] == clase), None)
    if i is not None:
        ax.semilogy(freqs, F[i] + 1e-4, lw=0.8, label=f"{clase} (rms {np.sqrt(np.mean(Vp[i]**2)):.2f} A)")
ax.set(xlabel="f [Hz]", ylabel="|I| [A]", title="vector rawfft de una ventana por clase"); ax.legend(fontsize=8)
plt.show()

Fc, nc = extraer_features(Vp, esc, CFG.clonar(estrategia="compact"), fs)
df_f = pd.DataFrame(Fc, columns=nc)
df_f["clase"] = [",".join(c for c, b in zip(ds.clases, y) if b) or "nada" for y in Yw]
df_f.groupby("clase")[["rms_A", "h1_A", "thd", "h3_rel", "par_impar", "centroide_hz", "cresta"]].median().round(3)

## 5. Baseline
`run_experiment` hace validación cruzada por captura (5 folds); las mezclas sintéticas solo entran al entrenamiento.
Todo el test son capturas **reales**: `ind` = individuales del fold de test, `comb` = combinadas reales (evaluadas con los
modelos de todos los folds, promediando probabilidades) y `todas` = ind + comb (rendimiento global del producto).
Métricas por ventana, por captura (media de probabilidades de sus ventanas, umbral 0.5) y por captura con umbrales por clase
aprendidos sobre las individuales out-of-fold (`+umbr`). El detalle por combinación muestra qué se predice para cada combinada real.

In [ ]:
LEADERBOARD = os.path.join(RUTA_SALIDAS, "leaderboard.csv")
res = run_experiment(ds, CFG, leaderboard=LEADERBOARD, etiqueta="base")
print(reporte(res.metricas, res.clases))
if "comb" in res.bloques:
    print("\nCombinadas reales por combinación (umbrales por clase):")
    print(reporte_combinaciones(res.bloques["comb"], res.clases, res.umbrales))

## 6. Barridos
Editá el diccionario `variantes` (cada entrada es un `CFG.clonar(...)`). Todo va al leaderboard; el gráfico muestra F1 vs costo del modelo (nodos → bytes en flash).

In [ ]:
variantes = {
    "jitter_2hz":     CFG.clonar(mix_jitter_fase=14.4),   # ±2 Hz sobre un ciclo de 50 Hz ≡ ±14.4°
    "fase_aleatoria": CFG.clonar(mix_fase="aleatoria"),
    "sin_mix":        CFG.clonar(mix_habilitado=False),   # ¿qué aportan las mezclas sintéticas?
    "mix400":         CFG.clonar(mix_total=400),
    "sin_agrupar":    CFG.clonar(agrupar={}),
    "compact":        CFG.clonar(estrategia="compact"),
    "fmax2000":       CFG.clonar(rawfft_fmax=2000.0),
    "fmax4000":       CFG.clonar(rawfft_fmax=4000.0),
    "fmax5000":       CFG.clonar(rawfft_fmax=5000.0),   # Nyquist
    "W4096":          CFG.clonar(W=4096),
    "hp20_causal":    CFG.clonar(filtro="hp", causal=True),
    "aug4":           CFG.clonar(aug_n=4),
    "rf40x8":         CFG.clonar(n_estimators=40),
    "rf20x12":        CFG.clonar(max_depth=12),
}
print(f"{'variante':14s} {'ind':>6s} {'comb':>6s} {'todas':>6s} {'todas+u':>8s} {'exact_c':>8s} {'nodos':>7s}")
for nombre, c in variantes.items():
    r = run_experiment(ds, c, leaderboard=LEADERBOARD, verbose=False, etiqueta=nombre)
    m = r.metricas
    print(f"{nombre:14s} {m['ind_c_f1_micro']:6.3f} {m.get('comb_c_f1_micro', float('nan')):6.3f} "
          f"{m['todas_c_f1_micro']:6.3f} {m.get('todas_u_f1_micro', float('nan')):8.3f} "
          f"{m.get('comb_c_exact_match', float('nan')):8.2f} {m['nodos_totales']:7d}  {m['segundos']:.0f} s")

In [ ]:
lb = cargar_leaderboard(LEADERBOARD, ordenar_por="todas_c_f1_micro")
cols = ["etiqueta", "nombre", "ind_c_f1_micro", "comb_c_f1_micro", "comb_c_exact_match", "todas_c_f1_micro", "todas_u_f1_micro", "nodos_totales", "segundos"]
display(lb[[c for c in cols if c in lb.columns]].head(15))
plots.plot_leaderboard(lb, y="todas_c_f1_micro", y2="comb_c_f1_micro")
plt.show()

## 7. Evaluación del mejor
Elegí `MEJOR` a mano (por F1 en `todas`/`comb` **y** por `nodos_totales`) o dejá la primera fila del leaderboard.
Ojo: el leaderboard mezcla experimentos con y sin `agrupar`; sus clases no son comparables una a una.
LOGO (leave-one-capture-out) es la evaluación definitiva de individuales (61 entrenamientos: ~2 min con `compact`, ~15 min con `rawfft`); las matrices por clase y los umbrales óptimos salen de las predicciones out-of-fold.

In [ ]:
MEJOR = Config.from_json(lb.iloc[0]["config"])
print("MEJOR:", MEJOR.nombre())
res_m = run_experiment(ds, MEJOR, verbose=False)
print(reporte(res_m.metricas, res_m.clases))
res_logo = run_experiment(ds, MEJOR.clonar(cv="logo"), verbose=False)
print("\nLOGO individuales por captura:",
      {k: round(res_logo.metricas[f"ind_c_{k}"], 3) for k in ("exact_match", "f1_micro", "f1_macro")})

In [ ]:
b = res_m.bloques["ind"]
Pg, Yg, _ = agregar_por_grupo(b["P"], b["Y"], b["grupos"])
umbrales = res_m.umbrales
print("umbrales por clase:", dict(zip(res_m.clases, np.round(umbrales, 2).tolist())))
if "comb" in res_m.bloques:
    print("\nCombinadas reales por combinación (umbrales por clase):")
    print(reporte_combinaciones(res_m.bloques["comb"], res_m.clases, umbrales))
plots.plot_matrices_por_clase(Yg, Pg, umbrales, res_m.clases)
plots.plot_importancias(res_m.importancias)
plt.show()

## 8. Export al ESP32-S3 y test de paridad
Entrena con todas las individuales (+ mezclas + augment), genera `nilm_modelo.h`, `nilm_config.h`, `pipeline_config.json` y `modelo.joblib`, y compila un programa C en el host que debe reproducir `predict_proba` de sklearn (diferencia < 1e-5).

In [ ]:
modelo, nombres_f, clases = entrenar_final(ds, MEJOR)
CARPETA_EXPORT = os.path.join(RUTA_SALIDAS, f"export_{MEJOR.nombre()}")
info = exportar(modelo, MEJOR, fs, nombres_f, clases, CARPETA_EXPORT, umbrales=umbrales)
print(info)
F_all, _, _, _ = armar_conjunto(sen, Y, G, MEJOR, fs)
paridad = test_paridad(CARPETA_EXPORT, F_all, predecir_proba(modelo, F_all, len(clases)))
print(paridad)
assert paridad["paridad_ok"], "el C exportado NO reproduce a sklearn: no flashear"
rob = robustez_snr(modelo, ds, MEJOR)
print("robustez SNR:", {k: round(v, 3) for k, v in rob.items()})
plots.plot_robustez(rob)
plt.show()
print("archivos en", CARPETA_EXPORT, ":", sorted(os.listdir(CARPETA_EXPORT)))

## 9. (Opcional) Modelos de referencia
No exportables al borde (o muy lentos); sirven para saber cuánto margen hay. `hgb` tarda varios minutos.

In [ ]:
CORRER_REFERENCIA = False
if CORRER_REFERENCIA:
    for m in ("knn", "logreg", "svm", "hgb"):
        r = run_experiment(ds, MEJOR.clonar(modelo=m), leaderboard=LEADERBOARD, verbose=False)
        print(f"{m:7s} ind={r.metricas['ind_c_f1_micro']:.3f} comb={r.metricas.get('comb_c_f1_micro', float('nan')):.3f} "
              f"todas={r.metricas['todas_c_f1_micro']:.3f} {r.metricas['segundos']:.0f} s")